<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/COM_SCI_M148_NN_no_genre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [8]:
train_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv"
validation_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv"

train_data = pd.read_csv(train_filepath)
validation_data = pd.read_csv(train_filepath)

In [9]:
class SpotifyDataset(Dataset):
  def __init__(self, csv_file):
    self.data = pd.read_csv(csv_file) # load data
    self.data['explicit'] = self.data['explicit'].astype(int) # change T/F to 1/0 encoding
    self.data = self.data.drop(columns=['track_genre']) # drop to reduce complexity rn
    self.x = self.data.drop(columns=['popularity']).values
    self.y = self.data['popularity'].values
    scaler = StandardScaler()
    self.x = scaler.fit_transform(self.x) #scale data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    features = self.x[idx]
    label = self.y[idx]
    features_tensor = torch.tensor(features, dtype=torch.float32)
    label_tensor = torch.tensor(label, dtype=torch.float32)
    return features_tensor, label_tensor

In [12]:
spotify_dataset = SpotifyDataset(csv_file=train_filepath)

In [13]:
# spotify to tensor verification

if __name__ == "__main__":
    train_loader = DataLoader(dataset=spotify_dataset, batch_size=32, shuffle=True)
    data_iter = iter(train_loader)
    features, labels = next(data_iter)

    print(f"Features batch shape: {features.shape}")
    print(f"Labels batch shape: {labels.shape}")

Features batch shape: torch.Size([32, 14])
Labels batch shape: torch.Size([32])


# NN

In [18]:
model = nn.Sequential(
    nn.Linear(14,32),
    nn.ReLU(),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1)
)

loss_type = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50

for epoch in range(num_epochs):
  epoch_loss = 0.0

  for features, labels in train_loader:
    # forward pass, predicted
    predictions = model(features)
    # loss
    loss = loss_type(predictions, labels.view(-1,1))
    #backprop, gradients and update weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    #update loss
    epoch_loss += loss.item()

  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader)}")

print("Training finished")

Epoch 1/50, Loss: 588.3334420015794
Training finished
Epoch 2/50, Loss: 483.74048942715825
Training finished
Epoch 3/50, Loss: 480.75054666544804
Training finished
Epoch 4/50, Loss: 479.5656079170992
Training finished
Epoch 5/50, Loss: 477.7966187643783
Training finished
Epoch 6/50, Loss: 475.02170808904543
Training finished
Epoch 7/50, Loss: 472.48749114611525
Training finished
Epoch 8/50, Loss: 470.7884078790138
Training finished
Epoch 9/50, Loss: 469.47663240324255
Training finished
Epoch 10/50, Loss: 468.4515558773384
Training finished
Epoch 11/50, Loss: 467.45674105686635
Training finished
Epoch 12/50, Loss: 466.744395716607
Training finished
Epoch 13/50, Loss: 465.9203898795758
Training finished
Epoch 14/50, Loss: 464.86271559390804
Training finished
Epoch 15/50, Loss: 463.8719383405431
Training finished
Epoch 16/50, Loss: 463.11088909161003
Training finished
Epoch 17/50, Loss: 462.43491699629016
Training finished
Epoch 18/50, Loss: 461.80892400731716
Training finished
Epoch 19/5

- Next step is to include track genre